# Data Preprocessing and Training Data Development

## Objective
The goal of this project is to preprocess the Nintendo Games dataset in preparation for machine learning modeling.

We will:

Convert categorical variables into numeric format
Standardize numeric features
Split the dataset into training and testing sets
Save a cleaned dataset for modeling


In [ ]:
##Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
##Load Dataset

In [2]:
df = pd.read_csv(r"C:\Users\kedha\Downloads\NintendoGames.csv")
df.head()

,meta_score,title,platform,date,user_score,link,esrb_rating,developers,genres
0,NaN,Super Mario RPG,Switch,"Nov 17, 2023",NaN,/game/switch/super-mario-rpg,E,['Nintendo'],"['Role-Playing', 'Japanese-Style']"
1,NaN,WarioWare: Move It!,Switch,"Nov 3, 2023",NaN,/game/switch/warioware-move-it!,RP,['Intelligent Systems'],"['Miscellaneous', 'Party / Minigame']"
2,NaN,Super Mario Bros. Wonder,Switch,"Oct 20, 2023",NaN,/game/switch/super-mario-bros-wonder,E,['Nintendo'],"['Action', 'Platformer', '2D']"
3,NaN,Detective Pikachu Returns,Switch,"Oct 6, 2023",NaN,/game/switch/detective-pikachu-returns,NaN,['Creatures Inc.'],"['Adventure', '3D', 'Third-Person']"
4,NaN,Fae Farm,Switch,"Sep 8, 2023",NaN,/game/switch/fae-farm,E10+,['Phoenix Labs'],"['Simulation', 'Virtual', 'Virtual Life']"


In [ ]:
##Dataset Overview

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1094 entries, 0 to 1093
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   meta_score   709 non-null    float64
 1   title        1094 non-null   str    
 2   platform     1094 non-null   str    
 3   date         1094 non-null   str    
 4   user_score   856 non-null    float64
 5   link         1094 non-null   str    
 6   esrb_rating  972 non-null    str    
 7   developers   1091 non-null   str    
 8   genres       1094 non-null   str    
dtypes: float64(2), str(7)
memory usage: 77.1 KB


In [4]:
df.describe()

,meta_score,user_score
count,709.000000,856.000000
mean,76.083216,7.671612
std,10.610735,1.031266
min,37.000000,3.100000
25%,69.000000,7.200000
50%,77.000000,7.900000
75%,83.000000,8.400000
max,99.000000,9.600000


In [5]:
df.isnull().sum()

meta_score     385
title            0
platform         0
date             0
user_score     238
link             0
esrb_rating    122
developers       3
genres           0
dtype: int64

Notes:
Categorical columns include: title, platform, ESRB rating, developers, genres
Numeric columns include: meta score, user score

In [ ]:
#Handle Missing Values

In [6]:
df['meta_score'] = df['meta_score'].fillna(df['meta_score'].median()) 
df['user_score'] = df['user_score'].fillna(df['user_score'].median())

df['esrb_rating'] = df['esrb_rating'].fillna(df['esrb_rating'].mode()[0])
df['developers'] = df['developers'].fillna("Unknown")

We remove missing values to ensure the dataset is clean before modeling.

In [ ]:
##Remove Unnecessary Columns

In [7]:
df = df.drop(columns=['title', 'link', 'date'])

In [ ]:
Columns such as title, link, and date are not useful for predictive modeling and are removed.

In [ ]:
##Examine Categorical Cardinality

In [8]:
categorical_cols = ['platform', 'esrb_rating', 'developers', 'genres'] 
for col in categorical_cols: 
    print(f"\n{col}") 
    print(df[col].value_counts().head(10)) 
    print("Unique Categories:", df[col].nunique())


platform
platform
3DS       259
Switch    209
DS        196
WII       188
WIIU       80
GBA        64
GC         52
N64        31
iOS        14
TG16)       1
Name: count, dtype: int64
Unique Categories: 10

esrb_rating
esrb_rating
E       782
T       150
E10+    142
M        15
RP        5
Name: count, dtype: int64
Unique Categories: 5

developers
developers
['Nintendo']                     354
['Intelligent Systems']           95
['Game Freak']                    45
['HAL Labs']                      34
['Level 5']                       27
['Omega Force']                   16
['Monolith Soft']                 15
['Camelot Software Planning']     15
['Rare Ltd.']                     15
['Arika']                         11
Name: count, dtype: int64
Unique Categories: 212

genres
genres
['Action', 'Platformer', '2D']                                 71
['Strategy', 'Turn-Based', 'Fantasy', 'Fantasy', 'Tactics']    34
['Strategy', 'Turn-Based', 'Tactics']                          31
['Acti

In [ ]:
Why Cardinality Matters

High-cardinality categorical variables create thousands of dummy columns after one-hot encoding. This increases:

Memory usage
Model complexity
Risk of overfitting

Therefore, rare categories should be grouped before encoding.

In [ ]:
##Reduce High Cardinality

In [10]:
top_developers = df['developers'].value_counts().nlargest(15).index
df['developers'] = df['developers'].apply( 
    lambda x: x if x in top_developers else 'Other' )
top_genres = df['genres'].value_counts().nlargest(15).index
df['genres'] = df['genres'].apply( 
    lambda x: x if x in top_genres else 'Other' )


In [ ]:
Rare developer and genre categories are grouped into "Other".

In [ ]:
##Encode Categorical Variables

In [11]:
df_encoded = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)
df_encoded.head()

,meta_score,user_score,platform_DS,platform_GBA,platform_GC,platform_N64,platform_Switch,platform_TG16),platform_WII,platform_WIIU,...,"genres_['Action', 'Platformer', 'Platformer', '2D', '2D']","genres_['Miscellaneous', 'General', 'General']","genres_['Miscellaneous', 'Party / Minigame']","genres_['Miscellaneous', 'Puzzle', 'Puzzle', 'General', 'Puzzle', 'General']","genres_['Role-Playing', 'Action RPG']","genres_['Role-Playing', 'Console-style RPG']","genres_['Role-Playing', 'Japanese-Style']","genres_['Role-Playing', 'Trainer']","genres_['Strategy', 'Turn-Based', 'Fantasy', 'Fantasy', 'Tactics']","genres_['Strategy', 'Turn-Based', 'Tactics']"
0,77.0,7.9,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,True,False,False,False
1,77.0,7.9,False,False,False,False,True,False,False,False,...,False,False,True,False,False,False,False,False,False,False
2,77.0,7.9,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,77.0,7.9,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,77.0,7.9,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
One-hot encoding is applied after reducing cardinality.

In [ ]:
##Define Features and Target

In [12]:
X = df_encoded.drop(columns=['meta_score'])
y = df_encoded['meta_score']

In [ ]:
The target variable for this project will be meta_score.

In [ ]:
##Train-Test Split

In [13]:
X_train, X_test, y_train, y_test = train_test_split( 
    X, 
    y, 
    test_size=0.2, 
    random_state=42 
    )
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

Training set shape: (875, 44)
Testing set shape: (219, 44)


In [ ]:
##Scale Only Numeric Features

In [15]:
scaler = StandardScaler()
numeric_cols = ['user_score']
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [ ]:
##Save Processed Dataset

In [16]:
processed_df = pd.concat([X, y], axis=1)
processed_df.to_csv( 
    "nintendo_games_processed.csv", 
    index=False
      )

In [ ]:
##Summary

In [ ]:
Preprocessing Steps Completed

The Nintendo Games dataset was successfully prepared for machine learning modeling through the following steps:

Loaded and explored the dataset
Identified numeric and categorical variables
Handled missing values using imputation
Removed unnecessary identifier columns
Examined categorical feature cardinality
Reduced high-cardinality categories by grouping rare values
Applied one-hot encoding to categorical variables
Split the dataset into training and testing sets
Standardized only numeric features
Saved the processed dataset for future modeling